In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.animation as animation
from scipy import special
from numba import jit

import sys
sys.path.append('../')
from src.config import GROUP_PARAMS, calculate_velocity_grid, VELOCITY_SPACE, COLLISION_PARAMS
from src.moment_utils import calc_moment, invert

In [ ]:
# Ak_data = np.load('../simulation_data/Ak_t100_20250606_121942.npy')
# bk_data = np.load('../simulation_data/bk_t100_20250606_121942.npy')
# wxk_data = np.load('../simulation_data/wxk_t100_20250606_121942.npy')
# wyk_data = np.load('../simulation_data/wyk_t100_20250606_121942.npy')
# wzk_data = np.load('../simulation_data/wzk_t100_20250606_121942.npy')
groups = np.load('../simulation_data/groups100_20250616_013822.npy', allow_pickle=True)
# meta_data = np.load('../simulation_data/metadata_t100_20250522_011407.npy', allow_pickle=True)
# mu1 = np.zeros(5)
# mu2 = np.zeros(5)
# fI_part1 = 0
# fI_part2 = 0

cx_vec, cy_vec, cz_vec, cx, cy, cz = calculate_velocity_grid()
# combined_fI = np.zeros(len(cx_vec))
# test_sum = 0
# for i, group in enumerate(groups[-2]):

#     if i < 4:
#         mu1 += group.mu
#     else:
#         mu2 += group.mu

#     f = group.A * np.exp(-group.b * ((cx - group.wx)**2 + (cy - group.wy)**2 + (cz - group.wz)**2))
#     lb_cx, ub_cx = group.group_bounds['group_bounds_cx']
#     lb_cy, ub_cy = group.group_bounds['group_bounds_cy']
#     lb_cz, ub_cz = group.group_bounds['group_bounds_cz']

#     # combined_f[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz] += 
#     group_fI = np.trapz(np.trapz(np.trapz(f[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz], cz_vec[lb_cz:ub_cz], axis=2), cy_vec[lb_cy:ub_cy], axis=1), cx_vec[lb_cx:ub_cx], axis=0)
#     test_sum += group.mu[0]

#     # if i < 4:
#     #     fI_part1 += np.trapz(np.trapz(f[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz], cz_vec[lb_cz:ub_cz], axis=2), cy_vec[lb_cy:ub_cy], axis=1)
#     # else:
#     #     fI_part2 += np.trapz(np.trapz(f[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz], cz_vec[lb_cz:ub_cz], axis=2), cy_vec[lb_cy:ub_cy], axis=1)
# print(test_sum)
# A, b, wx, wy, wz = invert(mu1, [1.0, 0.0, 0.0, 0.0], {'ci_cx': -3, 'cf_cx': 0.0, 'ci_cy': -3, 'cf_cy': 3.0, 'ci_cz': -3, 'cf_cz': 3.0})
# print(invert(mu2, [1.0, 0.0, 0.0, 0.0], {'ci_cx': 0.0, 'cf_cx': 3.0, 'ci_cy': -3, 'cf_cy': 3.0, 'ci_cz': -3, 'cf_cz': 3.0}))

In [ ]:
plt.rc('text', usetex=True)
plt.rc('font', family='serif')
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["figure.dpi"] = 150
plt.ioff()

# plt.plot(np.trapz(np.trapz(A * np.exp(-b * ((cx - wx)**2 + (cy - wy)**2 + (cz - wz)**2)), cz_vec, axis=2), cy_vec, axis=1)[0:121])
# f0 = 1 / (np.pi**1.5) * np.exp(-1 * (cx**2 + cy**2 + cz**2))
# f0I = np.trapz(np.trapz(f0, cz_vec, axis=2), cy_vec, axis=1)
# plt.plot(f0I[0:121])
# plt.plot(combined_fI[0:121])
# plt.show()

In [ ]:
%matplotlib inline
fig = plt.figure(figsize=(6, 6))
ax1 = fig.add_subplot(111)

K = 1 - 0.4 * np.exp(-0/6)
f0 = 1 / (2 * K * (np.pi * K)**1.5) * (5 * K - 3 + 2 * (1 - K) / K * (cx**2 + cy**2 + cz**2)) * np.exp(-(cx**2 + cy**2 + cz**2) / K)
# f0 = 1 / (np.pi**1.5) * np.exp(-1 * (cx**2 + cy**2 + cz**2))
f0I = np.trapz(np.trapz(f0, cy_vec, axis=1), cx_vec, axis=0)
# f0 = 0.5 * (3 / np.pi)**1.5 * (np.exp(-3.0 * (cx - 1)**2) + np.exp(-3.0 * (cx + 1)**2)) * np.exp(-3 * (cy**2 + cz**2))
# f0_init = np.trapz(np.trapz(f0, cy_vec, axis=1), cx_vec, axis=0)

def animate(i):
    ax1.clear()
    mu1 = 0
    mu2 = 0
    
    # combined_f = np.zeros((VELOCITY_SPACE['num_cx'], VELOCITY_SPACE['num_cy'], VELOCITY_SPACE['num_cz']))
    for group in groups[i]:
        # f = group.A * np.exp(-group.b * ((cx - group.wx)**2 + (cy - group.wy)**2 + (cz - group.wz)**2))
        lb_cx, ub_cx = group.group_bounds['group_bounds_cx']
        lb_cy, ub_cy = group.group_bounds['group_bounds_cy']
        lb_cz, ub_cz = group.group_bounds['group_bounds_cz']

        # combined_f[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz] = f[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz]
        if lb_cx == 0:
            mu1 += group.mu
        else:
            mu2 += group.mu

    A1, b1, wx1, wy1, wz1 = invert(mu1, [1.0, 0.0, 0.0, 0.0], {'ci_cx': -3.0, 'cf_cx': 3.0, 'ci_cy': -3.0, 'cf_cy': 3.0, 'ci_cz': -3.0, 'cf_cz': 0.0})
    A2, b2, wx2, wy2, wz2 = invert(mu2, [1.0, 0.0, 0.0, 0.0], {'ci_cx': -3.0, 'cf_cx': 3.0, 'ci_cy': -3.0, 'cf_cy': 3.0, 'ci_cz': 0.0, 'cf_cz': 3.0})

    fI1 = np.trapz(np.trapz(A1 * np.exp(-b1 * ((cx - wx1)**2 + (cy - wy1)**2 + (cz - wz1)**2)), cy_vec, axis=1), cx_vec, axis=0)
    fI2 = np.trapz(np.trapz(A2 * np.exp(-b2 * ((cx - wx2)**2 + (cy - wy2)**2 + (cz - wz2)**2)), cy_vec, axis=1), cx_vec, axis=0)

    ax1.plot(cx_vec[0:121], fI1[0:121], color='purple', label='Group 1')
    ax1.plot(cx_vec[120:], fI2[120:], color='purple', label='Group 2')

    ax1.set_xlabel(r'$C_z$', fontsize=20)
    ax1.set_ylabel(r'f', fontsize=20)
    ax1.set_ylim(-0.02, 1.0)
    ax1.tick_params(axis='both',labelsize=16)
    # ax1.plot(cx_vec, f0_init, '--', color='red', label='Initial')
    ax1.plot(cx_vec, f0I, '--', color='black', label='Final')
    ax1.legend(fontsize=14)
    if i % 10 == 0: print(i)

anim = animation.FuncAnimation(fig, animate, frames=5, interval=100)
anim.save('3d_animation.gif')


In [ ]:
Ak = Ak_data[0, 0, 0, 0]
bk = bk_data[0, 0, 0, 0]
wxk = wxk_data[0, 0, 0, 0]
wyk = wyk_data[0, 0, 0, 0]
wzk = wzk_data[0, 0, 0, 0]

I0x = np.sqrt(np.pi / (4 * bk)) * (special.erf(np.sqrt(bk) * (GROUP_PARAMS['cf_cx'][0] - wxk)) - special.erf(np.sqrt(bk) * (GROUP_PARAMS['ci_cx'][0] - wxk)))
I0y = np.sqrt(np.pi / (4 * bk)) * (special.erf(np.sqrt(bk) * (GROUP_PARAMS['cf_cy'][0] - wyk)) - special.erf(np.sqrt(bk) * (GROUP_PARAMS['ci_cy'][0] - wyk)))
I0z = np.sqrt(np.pi / (4 * bk)) * (special.erf(np.sqrt(bk) * (GROUP_PARAMS['cf_cz'][0] - wzk)) - special.erf(np.sqrt(bk) * (GROUP_PARAMS['ci_cz'][0] - wzk)))

I1x = (np.exp(-bk * (GROUP_PARAMS['ci_cx'][0] - wxk)**2) - np.exp(-bk * (GROUP_PARAMS['cf_cx'][0] - wxk)**2)) / (2 * bk)
I1y = (np.exp(-bk * (GROUP_PARAMS['ci_cy'][0] - wyk)**2) - np.exp(-bk * (GROUP_PARAMS['cf_cy'][0] - wyk)**2)) / (2 * bk)
I1z = (np.exp(-bk * (GROUP_PARAMS['ci_cz'][0] - wzk)**2) - np.exp(-bk * (GROUP_PARAMS['cf_cz'][0] - wzk)**2)) / (2 * bk)

I2x = -np.sqrt(np.pi) / (2 * np.sqrt(bk)) * \
        ((np.exp(-bk * (GROUP_PARAMS['cf_cx'][0] - wxk)**2) * (GROUP_PARAMS['cf_cx'][0] - wxk))/np.sqrt(np.pi * bk) - (np.exp(-bk * (GROUP_PARAMS['ci_cx'][0] - wxk)**2) * (GROUP_PARAMS['ci_cx'][0] - wxk))/np.sqrt(np.pi * bk)) + \
            np.sqrt(np.pi)/(4 * np.sqrt(bk**3)) * (special.erf(np.sqrt(bk) * (GROUP_PARAMS['cf_cx'][0] - wxk)) - special.erf(np.sqrt(bk) * (GROUP_PARAMS['ci_cx'][0] - wxk)))
I2y = -np.sqrt(np.pi) / (2 * np.sqrt(bk)) * \
    ((np.exp(-bk * (GROUP_PARAMS['cf_cy'][0] - wyk)**2) * (GROUP_PARAMS['cf_cy'][0] - wyk))/np.sqrt(np.pi * bk) - (np.exp(-bk * (GROUP_PARAMS['ci_cy'][0] - wyk)**2) * (GROUP_PARAMS['ci_cy'][0] - wyk))/np.sqrt(np.pi * bk)) + \
        np.sqrt(np.pi)/(4 * np.sqrt(bk**3)) * (special.erf(np.sqrt(bk) * (GROUP_PARAMS['cf_cy'][0] - wyk)) - special.erf(np.sqrt(bk) * (GROUP_PARAMS['ci_cy'][0] - wyk)))
I2z = -np.sqrt(np.pi) / (2 * np.sqrt(bk)) * \
    ((np.exp(-bk * (GROUP_PARAMS['cf_cz'][0] - wzk)**2) * (GROUP_PARAMS['cf_cz'][0] - wzk))/np.sqrt(np.pi * bk) - (np.exp(-bk * (GROUP_PARAMS['ci_cz'][0] - wzk)**2) * (GROUP_PARAMS['ci_cz'][0] - wzk))/np.sqrt(np.pi * bk)) + \
        np.sqrt(np.pi)/(4 * np.sqrt(bk**3)) * (special.erf(np.sqrt(bk) * (GROUP_PARAMS['cf_cz'][0] - wzk)) - special.erf(np.sqrt(bk) * (GROUP_PARAMS['ci_cz'][0] - wzk)))

print('density:', Ak * I0x * I0y * I0z)
print('ux:', Ak * (I1x + wxk * I0x) * I0y * I0z)
print('uy:', Ak * I0x * (I1y + wyk * I0y) * I0z)
print('uz:', Ak * I0x * I0y * (I1z + wzk * I0z))
print('energy:', (Ak * (I2x + wxk**2 * I0x + 2 * wxk * I1x) * I0y * I0z) + (Ak * (I2y + 2 * wyk * I1y + wyk**2 * I0y) * I0x * I0z) + (Ak * (I2z + 2 * wzk * I1z + wzk**2 * I0z) * I0x * I0y))

In [ ]:
entropy_list = np.zeros(len(groups))
combined_f = np.zeros((VELOCITY_SPACE['num_cx'], VELOCITY_SPACE['num_cy'], VELOCITY_SPACE['num_cz']))

# for t in range(len(groups)):
    # print(t)
for group in groups[-20]:
    f = group.A * np.exp(-group.b * ((cx - group.wx)**2 + (cy - group.wy)**2 + (cz - group.wz)**2))
    lb_cx, ub_cx = group.group_bounds['group_bounds_cx']
    lb_cy, ub_cy = group.group_bounds['group_bounds_cy']
    lb_cz, ub_cz = group.group_bounds['group_bounds_cz']
            
    combined_f[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz] = f[lb_cx:ub_cx, lb_cy:ub_cy, lb_cz:ub_cz]

entropy_list[0] = -np.trapz(np.trapz(np.trapz(combined_f * np.log(combined_f, where=combined_f > 0), cz_vec, axis=2), cy_vec, axis=1), cx_vec, axis=0)
print(entropy_list[0])

fig = plt.figure(figsize=(6, 6))
ax1 = fig.add_subplot(111)
ax1.plot(np.linspace(0, COLLISION_PARAMS['n_t'] * COLLISION_PARAMS['dt'], len(entropy_list)), entropy_list, color='black')
ax1.set_xlabel('Time', fontsize=20)
ax1.set_ylabel('Entropy', fontsize=20)
ax1.tick_params(axis='both', labelsize=16)
plt.savefig('entropy_plot.png', bbox_inches='tight')


In [ ]:
K = 1 - 0.4 * np.exp(-0/6)
f0 = 1 / (2 * K * (np.pi * K)**1.5) * (5 * K - 3 + 2 * (1 - K) / K * (cx**2 + cy**2 + cz**2)) * np.exp(-(cx**2 + cy**2 + cz**2) / K)

param = {'num_groups_cx': 1,
    'num_groups_cy': 1,
    'num_groups_cz': 1,
    'ci_cx': np.array([-3.0]),
    'cf_cx': np.array([3.0]),
    'group_bounds_cx': np.array([[0, 241]]),
    'ci_cy': np.array([-3.0]),
    'cf_cy': np.array([3.0]),
    'group_bounds_cy': np.array([[0, 241]]),
    'ci_cz': np.array([-3.0]),
    'cf_cz': np.array([3.0]),
    'group_bounds_cz': np.array([[0, 241]])}
mu = calc_moment(f0, cx, cy, cz, cx_vec, cy_vec, cz_vec)
A, b, wx, wy, wz = invert(mu.reshape((1, 1, 1, 5)), 1, 0, 0, 0, param)

ftest = A * np.exp(-b * ((cx - wx)**2 + (cy - wy)**2 + (cz - wz)**2))
ftestI = np.trapz(np.trapz(ftest, cz_vec, axis=2), cy_vec, axis=1)
mu2 = calc_moment(f0[0:121], cx[0:121, :, :], cy[0:121, :, :], cz[0:121, :, :], cx_vec[0:121], cy_vec, cz_vec)
mu3 = calc_moment(ftest[0:121], cx[0:121, :, :], cy[0:121, :, :], cz[0:121, :, :], cx_vec[0:121], cy_vec, cz_vec)

param2 = {'num_groups_cx': 1,
    'num_groups_cy': 1,
    'num_groups_cz': 1,
    'ci_cx': np.array([-3.0]),
    'cf_cx': np.array([0.0]),
    'group_bounds_cx': np.array([[0, 121]]),
    'ci_cy': np.array([-3.0]),
    'cf_cy': np.array([3.0]),
    'group_bounds_cy': np.array([[0, 241]]),
    'ci_cz': np.array([-3.0]),
    'cf_cz': np.array([3.0]),
    'group_bounds_cz': np.array([[0, 241]])}
A, b, wx, wy, wz = invert(mu3.reshape((1, 1, 1, 5)), 1, 0, 0, 0, param2)
ftest2 = A * np.exp(-b * ((cx - wx)**2 + (cy - wy)**2 + (cz - wz)**2))
ftest2I = np.trapz(np.trapz(ftest2, cz_vec, axis=2), cy_vec, axis=1)

plt.plot(cx_vec, ftestI)
plt.plot(cx_vec[0:121], ftest2I[0:121])
plt.plot(cx_vec, np.trapz(np.trapz(f0, cz_vec, axis=2), cy_vec, axis=1))